# 3D reconstruction from fisheye photos with LiteCalib

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kocurvik/litecalib/blob/main/notebooks/reconstruction.ipynb)

Feed-forward reconstruction models such as [Pi3X](https://github.com/yyfz/Pi3) predict the camera poses and the 3D
geometry of a scene from a handful of images in one forward pass. They are trained on (nearly) pinhole images, so
photos from a fisheye lens confuse them: the straight walls of a room are curved in the images, and the model cannot
tell that from the scene itself.

This notebook calibrates the fisheye lens from ten photos with **LiteCalib**, undistorts the photos to a pinhole
camera, and gives Pi3X both the undistorted images and their intrinsics. It compares three reconstructions of the
same ten photos against the scene's ground-truth poses:

1. **fisheye**: the original photos, no intrinsics (what you can do without a calibration);
2. **LiteCalib**: the photos undistorted with LiteCalib's camera, plus its intrinsics;
3. **ground truth**: the photos undistorted with the scene's known camera, plus its intrinsics (the best a
   calibration can do).

The photos are from [Zip-NeRF](https://jonbarron.info/zipnerf/) (Barron et al., ICCV 2023; CC BY 4.0): indoor scenes
taken with a full-frame fisheye lens, released with a COLMAP reconstruction of every photo. Only the files needed are
read from the public archive, with HTTP range requests (tens of MB instead of the whole archive).

Use a GPU runtime (*Runtime > Change runtime type > GPU*).

In [ ]:
!pip install -q litecalib
!pip install -q git+https://github.com/yyfz/Pi3.git@b412c3bd236dfd7686f1e4b48004d5087f2fa093

## Settings

In [ ]:
SCENE = "berlin"          # alameda, berlin, london or nyc
START = 250               # index of the first of the consecutive photos
FRAMES = 10               # photos to reconstruct
CAM_ID = "division:1"     # the camera model LiteCalib fits: the one-parameter division model
PINHOLE_SIZE = (396, 696) # (h, w) of the undistorted images: the crop of Zip-NeRF's own undistorted release
WORK = "zipnerf_work"

## Download the photos and the ground truth

`RangeFile` makes a remote zip archive look like a local file, so Python's `zipfile` reads only the central directory
and the members it is asked for: the ten 876 x 584 photos, the COLMAP camera (`cameras.bin`, a Kannala-Brandt fisheye
model with four coefficients) and the COLMAP poses (`images.bin`, read until the ten photos' poses have been found).

In [ ]:
import io
import struct
import zipfile
from pathlib import Path

import numpy as np
import requests
import torch
from PIL import Image


class RangeFile(io.RawIOBase):
    # a remote file, read by HTTP range requests
    def __init__(self, url):
        self.url, self.pos, self.session = url, 0, requests.Session()
        self.size = int(self.session.head(url, timeout=60).headers["Content-Length"])

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else self.pos + off if whence == 1 else self.size + off
        return self.pos

    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        r = self.session.get(self.url, headers={"Range": f"bytes={self.pos}-{self.pos + n - 1}"}, timeout=120)
        r.raise_for_status()
        b[:n] = r.content
        self.pos += n
        return n


def qvec_to_R(q):
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])


def read_poses(f, stems):
    # COLMAP images.bin -> {stem: (R, t)} (world to camera) for the given image stems
    (n,) = struct.unpack("<Q", f.read(8))
    poses = {}
    for _ in range(n):
        _, qw, qx, qy, qz, tx, ty, tz, _ = struct.unpack("<i7di", f.read(64))
        name = b""
        while (ch := f.read(1)) != b"\x00":
            name += ch
        (npts,) = struct.unpack("<Q", f.read(8))
        f.read(24 * npts)
        stem = Path(name.decode()).stem
        if stem in stems:
            poses[stem] = (qvec_to_R((qw, qx, qy, qz)), np.array([tx, ty, tz]))
            if len(poses) == len(stems):
                break
    return poses


url = f"https://storage.googleapis.com/gresearch/refraw360/zipnerf/{SCENE}.zip"
archive = zipfile.ZipFile(io.BufferedReader(RangeFile(url), buffer_size=1 << 20))
blob = archive.read(f"{SCENE}/sparse/0/cameras.bin")
_, _, w_full, h_full = struct.unpack("<iiQQ", blob[8:32])
gt_full = np.array(struct.unpack("<8d", blob[32:96]))  # OPENCV_FISHEYE: fx, fy, cx, cy, k1, k2, k3, k4
level = w_full // 876
folder = f"{SCENE}/images_{level}/"
names = sorted(n for n in archive.namelist() if n.startswith(folder) and n.lower().endswith(".jpg"))
image_dir = Path(WORK, SCENE, "fisheye")
image_dir.mkdir(parents=True, exist_ok=True)
paths = []
for name in names[START : START + FRAMES]:
    path = image_dir / Path(name).name
    if not path.exists():
        path.write_bytes(archive.read(name))
    paths.append(path)
h, w = np.array(Image.open(paths[0])).shape[:2]
gt_params = gt_full.copy()
gt_params[:4] /= level  # COLMAP's pixel origin is the corner of the top-left pixel, as LiteCalib's
with archive.open(f"{SCENE}/sparse/0/images.bin") as f:
    gt_poses = read_poses(f, {p.stem for p in paths})
print(f"{SCENE}: {len(paths)} photos of {w} x {h}; GT camera kb:4 {np.round(gt_params, 4).tolist()}; "
      f"{len(gt_poses)} GT poses")

## Calibrate with LiteCalib

One camera is fitted jointly to the ten photos (`shared=True`): they all come from one lens. The result holds `K`,
the distortion coefficient and the camera model, and undistorts the photos itself. The ground truth is a different
camera model (Kannala-Brandt with four coefficients), so the two are compared by their fields of view.

In [ ]:
import litecalib

model = litecalib.load("litecalib_vitl")

In [ ]:
from litecalib import Calibration

calib = model.calibrate([str(p) for p in paths], CAM_ID, shared=True)
gt_calib = Calibration("kb:4", torch.tensor(gt_params)[None], (h, w), 1)
print(calib)
fov, gt_fov = calib.fov(), gt_calib.fov()
print(f"field of view, vertical / horizontal: LiteCalib {fov['vfov'][0]:.1f} / {fov['hfov'][0]:.1f} deg, "
      f"ground truth {gt_fov['vfov'][0]:.1f} / {gt_fov['hfov'][0]:.1f} deg")

The photos undistorted to a pinhole camera of `PINHOLE_SIZE` with the fitted focal length and the principal point at
the center. Straight lines in the room should now be straight.

In [ ]:
import matplotlib.pyplot as plt


def undistort_all(c, tag):
    # undistort the photos to PINHOLE_SIZE; returns the saved paths and the pinhole K
    out_dir = Path(WORK, SCENE, tag)
    out_dir.mkdir(parents=True, exist_ok=True)
    out = []
    for p, im in zip(paths, c.undistort([str(p) for p in paths], size=PINHOLE_SIZE)):
        arr = (im.permute(1, 2, 0).clamp(0, 1).numpy() * 255).round().astype(np.uint8)
        Image.fromarray(arr).save(out_dir / p.name, quality=95)
        out.append(out_dir / p.name)
    return out, c.undistorted_K(size=PINHOLE_SIZE)[0].numpy()


undistorted = undistort_all(calib, "litecalib")
gt_undistorted = undistort_all(gt_calib, "ground_truth")

fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
for col, j in enumerate((0, FRAMES // 2, FRAMES - 1)):
    axes[0, col].imshow(Image.open(paths[j]))
    axes[0, col].set_title("fisheye photo")
    axes[1, col].imshow(Image.open(undistorted[0][j]))
    axes[1, col].set_title("undistorted with LiteCalib")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()

## Reconstruct with Pi3X

Pi3X takes the images of a scene and, optionally, the intrinsics of each image. Every view is resized to Pi3's pixel
budget (at most 255k pixels, sides a multiple of 14) and center-cropped to one common size; a given `K` follows the
same resize and crop.

In [ ]:
from pi3.models.pi3x import Pi3X

pi3x = Pi3X.from_pretrained("yyfz233/Pi3X").to("cuda").eval()

In [ ]:
import math

import torch.nn.functional as F
from torchvision import transforms as TF


def pi3_size(aspect, limit=255_000):
    # the largest (w, h), multiples of 14, of this aspect ratio within the pixel budget
    s = math.sqrt(limit / aspect)
    k, m = round(s * aspect / 14), round(s / 14)
    while (k * 14) * (m * 14) > limit:
        if k / m > aspect:
            k -= 1
        else:
            m -= 1
    return k * 14, m * 14


def reconstruct(image_paths, K=None):
    # Pi3X on the images (with the pinhole K of every image, or none): world-to-camera poses per image stem, and
    # the predicted 3D points with their colors
    images = [Image.open(p).convert("RGB") for p in image_paths]
    tw, th = pi3_size(float(np.mean([im.size[0] / im.size[1] for im in images])))
    tensors, Ks = [], []
    for im in images:
        iw, ih = im.size
        s = max(tw / iw, th / ih)
        rw, rh = max(tw, round(s * iw)), max(th, round(s * ih))
        x0, y0 = round((rw - tw) / 2), round((rh - th) / 2)
        tensors.append(TF.ToTensor()(im.resize((rw, rh), Image.Resampling.LANCZOS).crop((x0, y0, x0 + tw, y0 + th))))
        if K is not None:
            sx, sy = rw / iw, rh / ih
            Ks.append([[K[0, 0] * sx, 0, K[0, 2] * sx - x0], [0, K[1, 1] * sy, K[1, 2] * sy - y0], [0, 0, 1]])
    imgs = torch.stack(tensors)[None].cuda()
    intrinsics = torch.tensor(Ks, dtype=torch.float32)[None].cuda() if K is not None else None
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=dtype):
        res = pi3x(imgs=imgs, intrinsics=intrinsics)
    c2w = res["camera_poses"][0].double().cpu().numpy()
    poses = {}
    for i, p in enumerate(image_paths):
        R = c2w[i, :3, :3].T
        poses[p.stem] = (R, -R @ c2w[i, :3, 3])
    keep = (torch.sigmoid(res["conf"][0, ..., 0]) > 0.1).cpu().numpy()
    points = res["points"][0].float().cpu().numpy()[keep]
    colors = imgs[0].permute(0, 2, 3, 1).float().cpu().numpy()[keep]
    return poses, (points, colors)

## Compare the camera poses with the ground truth

For every pair of the ten photos, the error is the larger of the relative rotation error and the angle between the
relative translation directions (the scale of a reconstruction is arbitrary). mAA(t) is the area under the
recall-error curve up to t degrees (higher is better).

In [ ]:
def pair_errors(est, gt, stems):
    errors = []
    for a in range(len(stems)):
        for b in range(a + 1, len(stems)):
            (ri, ti), (rj, tj) = est[stems[a]], est[stems[b]]
            (gi, si), (gj, sj) = gt[stems[a]], gt[stems[b]]
            r_est, r_gt = rj @ ri.T, gj @ gi.T
            t_est, t_gt = tj - r_est @ ti, sj - r_gt @ si
            rot = np.degrees(np.arccos(np.clip((np.trace(r_est @ r_gt.T) - 1) / 2, -1, 1)))
            cos = t_est @ t_gt / (np.linalg.norm(t_est) * np.linalg.norm(t_gt) + 1e-12)
            errors.append(max(rot, np.degrees(np.arccos(np.clip(cos, -1, 1)))))
    return errors


def maa(errors, threshold):
    e = np.sort(np.asarray(errors))
    recall = np.arange(1, len(e) + 1) / len(e)
    e, recall = np.r_[0.0, e], np.r_[0.0, recall]
    last = int(np.searchsorted(e, threshold))
    x, y = np.r_[e[:last], threshold], np.r_[recall[:last], recall[last - 1]]
    return float(np.sum((x[1:] - x[:-1]) * (y[1:] + y[:-1]) / 2) / threshold)


variants = {"fisheye": (paths, None), "LiteCalib": undistorted, "ground truth": gt_undistorted}
stems = [p.stem for p in paths]
errors, clouds = {}, {}
for v, (image_paths, K) in variants.items():
    poses, clouds[v] = reconstruct(image_paths, K)
    errors[v] = pair_errors(poses, gt_poses, stems)

print(f"{'':14s} {'mAA(5)':>8s} {'mAA(10)':>8s} {'median error':>13s}")
for v, e in errors.items():
    print(f"{v:14s} {100 * maa(e, 5):8.1f} {100 * maa(e, 10):8.1f} {np.median(e):11.2f} deg")

## Look at the reconstructions

The point clouds Pi3X predicts without and with the calibration. Drag to rotate; walls and the floor should be flat.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

shown = ["fisheye", "LiteCalib"]
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}] * 2], subplot_titles=shown)
rng = np.random.default_rng(0)
for col, v in enumerate(shown, start=1):
    pts, rgb = clouds[v]
    keep = rng.choice(len(pts), min(len(pts), 60_000), replace=False)
    pts, rgb = pts[keep], np.clip(rgb[keep], 0, 1)
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=-pts[:, 1], z=-pts[:, 2], mode="markers",
                               marker=dict(size=1.2, color=[f"rgb({r * 255:.0f},{g * 255:.0f},{b * 255:.0f})"
                                                            for r, g, b in rgb])), row=1, col=col)
fig.update_layout(height=550, showlegend=False, margin=dict(l=0, r=0, t=30, b=0))
for scene in ("scene", "scene2"):
    fig.layout[scene].update(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)
fig.show()

## Your own photos

```python
calib = model.calibrate(["img1.jpg", "img2.jpg", ...], "division:1", shared=True)   # or kb:4, radial:2, ...
images = calib.undistort(["img1.jpg", "img2.jpg", ...], scale=0.8)                   # zoom out to keep more of the view
K = calib.undistorted_K(scale=0.8)                                                    # the undistorted images' intrinsics
```